# Kaggle 01 - Build Embeddings / Qdrant Index

Use this notebook on Kaggle GPU to build the real Qdrant Cloud index.

Kaggle settings:
- Accelerator: GPU T4/P100 or better
- Internet: On
- Secrets: `QDRANT_URL`, `QDRANT_API_KEY`, optional `OPENROUTER_API_KEY`

Recommended strategy:
1. Run diagnostics.
2. Run a small resumable index job.
3. Increase limits or remove limits after the first successful run.


In [ ]:
REPO_URL = "https://github.com/phamdinhhai/project-ks2.git"
PROJECT_DIR = "/kaggle/working/project-ks2"

import os
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git pull --ff-only


In [ ]:
!python -m pip install -U pip
!pip install -e ".[gpu,qdrant,agent,eval]"
!pip install requests accelerate bitsandbytes qwen-vl-utils


In [ ]:
# Load Kaggle Secrets. Add these in Notebook > Add-ons > Secrets.
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ["OPENROUTER_API_KEY", "QDRANT_URL", "QDRANT_API_KEY"]:
    try:
        value = secrets.get_secret(name)
    except Exception as exc:
        value = None
        print(f"Secret {name} unavailable: {exc}")
    if value:
        os.environ[name] = value

os.environ.setdefault("OPENROUTER_MODEL", "google/gemini-2.5-flash")
os.environ.setdefault("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
os.environ.setdefault("QDRANT_TEXT_COLLECTION", "text_chunks_prod")
os.environ.setdefault("QDRANT_IMAGE_COLLECTION", "image_patches_prod")
os.environ.setdefault("QDRANT_INDEX_STATE", "/kaggle/working/outputs/index_state/kaggle_index_state.json")

print("OPENROUTER_API_KEY set:", bool(os.environ.get("OPENROUTER_API_KEY")))
print("OPENROUTER_MODEL:", os.environ.get("OPENROUTER_MODEL"))
print("QDRANT_URL set:", bool(os.environ.get("QDRANT_URL")))
print("QDRANT_API_KEY set:", bool(os.environ.get("QDRANT_API_KEY")))
print("Text collection:", os.environ.get("QDRANT_TEXT_COLLECTION"))
print("Image collection:", os.environ.get("QDRANT_IMAGE_COLLECTION"))


In [ ]:
import torch, os
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("Working dir:", os.getcwd())


In [ ]:
# Qdrant connectivity check
!python -m medical_rag test-qdrant --qdrant-url "$QDRANT_URL" --use-cloud-auth


In [ ]:
# Optional real model smoke test. Skip if you only want to start indexing.
!python -m medical_rag test-encoders --no-mock --include-bge


In [ ]:
# Safe dry run: no points are uploaded.
!python scripts/colab_workflow.py build-index-resumable   --data-dir data   --qdrant-url "$QDRANT_URL"   --datasets all   --modality both   --image-mode full_only   --max-records 20   --dry-run   --use-cloud-auth


In [ ]:
# First real resumable job. Increase limits after this succeeds.
!python scripts/colab_workflow.py build-index-resumable   --data-dir data   --qdrant-url "$QDRANT_URL"   --datasets all   --modality both   --image-mode full_only   --max-records 1000   --max-minutes 100   --batch-size 16   --use-cloud-auth


In [ ]:
# Verify Qdrant point counts after indexing.
!python -m medical_rag test-qdrant --qdrant-url "$QDRANT_URL" --use-cloud-auth


In [ ]:
# Package state/output files for download from Kaggle Output panel.
!mkdir -p /kaggle/working/artifacts
!cp -r /kaggle/working/project-ks2/outputs /kaggle/working/artifacts/outputs || true
!find /kaggle/working/artifacts -maxdepth 3 -type f | head -50
